# Multimodal Diagnosis Assistant
This assistant takes medical images and patient text descriptions (e.g., symptoms, history) to provide diagnostic suggestions.

In [ ]:
# === Install Required Packages ===
!pip install gradio transformers torch torchvision torchaudio timm

In [ ]:
# === Imports ===
import gradio as gr
from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline
from PIL import Image
import torch
import requests
from io import BytesIO

In [ ]:
# === Load Vision-Language Model (BLIP for image captioning) ===
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base')
model.eval()

In [ ]:
# === Diagnostic Reasoning LLM Pipeline (text + caption) ===
reasoning_pipeline = pipeline('text2text-generation', model='google/flan-t5-large')

In [ ]:
# === Multimodal Diagnosis Function ===
def diagnose(image, symptoms):
    image = Image.open(image).convert('RGB')
    inputs = processor(image, return_tensors='pt')
    caption_ids = model.generate(**inputs)
    caption = processor.decode(caption_ids[0], skip_special_tokens=True)
    prompt = f"Image suggests: {caption}. Patient symptoms: {symptoms}. What are the likely diagnoses?"
    response = reasoning_pipeline(prompt, max_length=256)[0]['generated_text']
    return f"🖼 Caption: {caption}\n🩺 Diagnosis: {response}"

In [ ]:
# === Gradio Interface ===
gr.Interface(
    fn=diagnose,
    inputs=[gr.Image(type='filepath'), gr.Textbox(lines=3, label='Enter Symptoms')],
    outputs=gr.Textbox(label='Diagnosis Result'),
    title='Multimodal Diagnosis Assistant',
    description='Upload a medical image and describe symptoms to get a diagnosis suggestion.'
).launch()